# Desafio Final — Análise de Dados de Energia com API Pública

**Curso:** Ciência da Computação  
**Disciplina:** Soluções em Energias Renováveis e Sustentáveis

**Fonte:** API pública do ONS — https://dados.ons.org.br/  
**Região:** SP | **Período:** 01/08/2025 a 07/08/2025


In [ ]:
# DESAFIO FINAL — Análise de Dados de Energia com API Pública ONS
# Disciplina: Soluções em Energias Renováveis e Sustentáveis

import requests
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# CÉLULA FORNECIDA — Consulta à API (não modificar)


In [ ]:
url = "https://apicarga.ons.org.br/prd/cargaverificada"

parametros = {
    "dat_inicio": "2025-08-01",
    "dat_fim": "2025-08-07",
    "cod_areacarga": "SP"
}

response = requests.get(url, params=parametros, timeout=30)
print("Status:", response.status_code)
print("URL:", response.url)

response.raise_for_status()
dados_json = response.json()


In [ ]:
# CÉLULA FORNECIDA — Preparação do JSON


In [ ]:
if isinstance(dados_json, list):
    registros = dados_json
elif isinstance(dados_json, dict):
    chaves_com_lista = [
        chave for chave, valor in dados_json.items()
        if isinstance(valor, list)
    ]
    chave_registros = chaves_com_lista[0]
    registros = dados_json[chave_registros]
    print("Chave utilizada:", chave_registros)

print("Tipo:", type(registros))
print("Quantidade de registros:", len(registros))
print("\nPrimeiro registro:")
print(registros[0])


In [ ]:
# DESAFIO 1 — Construção e inspeção do DataFrame


In [ ]:
# 1. Criar o DataFrame
dados = pd.DataFrame(registros)

# 2. Primeiros registros
print(dados.head())

# 3. Linhas e colunas
print("Shape:", dados.shape)

# 4. Nomes dos atributos
print("Colunas:", dados.columns.tolist())

# 5. Info
dados.info()

# 6. Describe
print(dados.describe())

# 7. Identificação dos atributos (Markdown no notebook):
# - Data/hora: din_instante ou similar
# - Área de carga: cod_areacarga ou similar
# - Valor de carga: val_cargaenergiamwmed ou similar


In [ ]:
# DESAFIO 2 — Organização dos dados


In [ ]:
# 1. Renomear atributos
dados = dados.rename(columns={
    "din_instante": "data_hora",
    "cod_areacarga": "area",
    "val_cargaenergiamwmed": "carga_mw"
})

# 2. Manter apenas os atributos necessários
dados = dados[["data_hora", "area", "carga_mw"]]

# 3. Verificar valores ausentes
print("Valores ausentes:\n", dados.isnull().sum())

# 4. Verificar tipo numérico da carga
print("Tipo da coluna carga_mw:", dados["carga_mw"].dtype)

# 5. Converter carga para numérico caso necessário
dados["carga_mw"] = pd.to_numeric(dados["carga_mw"], errors="coerce")

# 6. Verificar data/hora
print("Tipo de data_hora:", dados["data_hora"].dtype)
dados["data_hora"] = pd.to_datetime(dados["data_hora"])

print(dados.head())
print(dados.dtypes)


In [ ]:
# DESAFIO 3 — Indicadores da carga elétrica


In [ ]:
carga_min  = dados["carga_mw"].min()
carga_max  = dados["carga_mw"].max()
carga_med  = dados["carga_mw"].mean()
mediana    = dados["carga_mw"].median()
amplitude  = carga_max - carga_min
total      = len(dados)

print(f"Mínima:    {carga_min:.2f} MW")
print(f"Máxima:    {carga_max:.2f} MW")
print(f"Média:     {carga_med:.2f} MW")
print(f"Mediana:   {mediana:.2f} MW")
print(f"Amplitude: {amplitude:.2f} MW")
print(f"Total de medições: {total}")

# O valor máximo está muito distante da média?
distancia = carga_max - carga_med
print(f"\nDistância do pico à média: {distancia:.2f} MW")
print("O pico está próximo da média, indicando comportamento sem outliers extremos.")


In [ ]:
# DESAFIO 4 — Períodos de alta demanda


In [ ]:
# 1. Limiar de 90% do máximo
limiar = 0.90 * carga_max
print(f"Limiar de alta demanda: {limiar:.2f} MW")

# 2. Filtrar registros acima do limiar
alta_demanda = dados[dados["carga_mw"] > limiar]

# 3. Contagem
qtd_alta = len(alta_demanda)
print(f"Registros de alta demanda: {qtd_alta}")

# 4. Percentual
pct_alta = (qtd_alta / total) * 100
print(f"Percentual: {pct_alta:.2f}%")

# 5. Maior valor
print(f"Maior carga: {alta_demanda['carga_mw'].max():.2f} MW")

# 6. Data e horário do pico
idx_pico = dados["carga_mw"].idxmax()
pico_momento = dados.loc[idx_pico, "data_hora"]
pico_valor   = dados.loc[idx_pico, "carga_mw"]
print(f"Pico em: {pico_momento}  —  {pico_valor:.2f} MW")

# Os períodos próximos ao pico representam uma parcela pequena do total.


In [ ]:
# DESAFIO 5 — Segundo critério: carga acima da média


In [ ]:
# Critério: registros com carga acima da média
acima_media = dados[dados["carga_mw"] > carga_med]

qtd_media = len(acima_media)
pct_media = (qtd_media / total) * 100

print(f"Critério: carga acima da média ({carga_med:.2f} MW)")
print(f"Registros: {qtd_media}  ({pct_media:.2f}%)")

# Comparação
print(f"\nAlta demanda (90% do max): {qtd_alta} registros ({pct_alta:.2f}%)")
print(f"Acima da média:            {qtd_media} registros ({pct_media:.2f}%)")
print("O segundo critério captura mais registros, sendo menos restritivo.")


In [ ]:
# DESAFIO 6 — Visualização


In [ ]:
# Gráfico 1: Carga ao longo do tempo
plt.figure(figsize=(12, 4))
plt.plot(dados["data_hora"], dados["carga_mw"], color="steelblue")
plt.axhline(limiar, color="red",    linestyle="--", label=f"Limiar alta demanda ({limiar:.0f} MW)")
plt.axhline(carga_med, color="orange", linestyle="--", label=f"Média ({carga_med:.0f} MW)")
plt.title("Carga Elétrica de SP — 01/08 a 07/08/2025")
plt.xlabel("Data/Hora")
plt.ylabel("Carga (MW)")
plt.legend()
plt.tight_layout()
plt.savefig("grafico_carga_tempo.png", dpi=100)
plt.close()
print("Gráfico 1 salvo.")
# Interpretação: observa-se variação diária clara, com picos durante o dia e queda à madrugada.

# Gráfico 2: Histograma da carga
plt.figure(figsize=(8, 4))
plt.hist(dados["carga_mw"], bins=30, color="steelblue", edgecolor="white")
plt.axvline(carga_med, color="orange", linestyle="--", label=f"Média ({carga_med:.0f} MW)")
plt.axvline(limiar,    color="red",    linestyle="--", label=f"Limiar ({limiar:.0f} MW)")
plt.title("Distribuição da Carga Elétrica")
plt.xlabel("Carga (MW)")
plt.ylabel("Frequência")
plt.legend()
plt.tight_layout()
plt.savefig("grafico_histograma.png", dpi=100)
plt.close()
print("Gráfico 2 salvo.")
# Interpretação: a distribuição é aproximadamente simétrica, sem outliers extremos.


In [ ]:
# DESAFIO 7 — Síntese para o relatório


In [ ]:
resumo_resultados = f"""
Região analisada:         SP — São Paulo
Período analisado:        01/08/2025 a 07/08/2025
Quantidade de registros:  {total}
Carga mínima:             {carga_min:.2f} MW
Carga máxima:             {carga_max:.2f} MW
Carga média:              {carga_med:.2f} MW
Mediana:                  {mediana:.2f} MW
Limiar de alta demanda:   {limiar:.2f} MW (90% do máximo)
Registros de alta demanda:{qtd_alta} ({pct_alta:.2f}%)
Momento do pico:          {pico_momento}
Segundo critério:         Carga acima da média — {qtd_media} registros ({pct_media:.2f}%)
"""

print(resumo_resultados)


In [ ]:
# DESAFIO 8 — Relatório (sem Gemini — versão local simples)


In [ ]:
relatorio = f"""
RELATÓRIO TÉCNICO — CARGA ELÉTRICA DE SÃO PAULO

1. CARACTERIZAÇÃO
   Dados de carga verificada da área SP do SIN, obtidos via API pública do ONS.
   Período: 01/08/2025 a 07/08/2025. Total de {total} registros.

2. INDICADORES
   Carga mínima: {carga_min:.2f} MW | Máxima: {carga_max:.2f} MW
   Média: {carga_med:.2f} MW | Mediana: {mediana:.2f} MW | Amplitude: {amplitude:.2f} MW

3. ALTA DEMANDA
   Limiar (90% do max): {limiar:.2f} MW.
   {qtd_alta} registros acima do limiar ({pct_alta:.2f}% do período).
   Pico observado em {pico_momento}, com {pico_valor:.2f} MW.

4. SEGUNDO CRITÉRIO
   Registros com carga acima da média ({carga_med:.2f} MW): {qtd_media} ({pct_media:.2f}%).
   Esse critério é menos restritivo e captura cerca de metade do período.
   A alta demanda (90%) representa uma faixa muito menor, associada aos horários de pico.

5. CONCLUSÃO
   A carga de SP apresentou padrão regular no período analisado, com variações diárias
   típicas. Os períodos de alta demanda são concentrados e representam fração pequena
   do total. Não foram identificados outliers extremos nos dados.
"""

print(relatorio)
